In [7]:
from flask import Flask, render_template, request, jsonify
import cv2
import numpy as np
from werkzeug.utils import secure_filename
import os
from tensorflow.keras.models import load_model


In [ ]:

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = 'static/uploads'
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 16MB limit

# Load CNN model
cnn_model = load_model('Models/ASL_alphabet_models/cnn_alpha_final.h5')

CLASS_NAMES = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 
               'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [8]:

def process_cnn_prediction(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (64, 64))  
    img = img.astype('float32') / 255.0
    img = np.expand_dims(img, axis=0)
    
    pred = cnn_model.predict(img)[0]
    class_index = np.argmax(pred)
    confidence = float(pred[class_index]) * 100
    
    return CLASS_NAMES[class_index], confidence


In [10]:

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    if 'file' not in request.files:
        return jsonify({'error': 'No file uploaded'})
    
    file = request.files['file']
    if file.filename == '':
        return jsonify({'error': 'No selected file'})
    
    # Validate file type
    if not file.filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        return jsonify({'error': 'Invalid file type. Please upload an image.'})
    
    try:
        # Save uploaded file
        filename = secure_filename(file.filename)
        filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(filepath)
        
        # Process with CNN model
        prediction, confidence = process_cnn_prediction(filepath)
        
        return jsonify({
            'success': True,
            'prediction': prediction,
            'confidence': f"{confidence:.2f}%",
            'image_url': filepath
        })
    except Exception as e:
        return jsonify({'error': f'Processing error: {str(e)}'})


In [13]:
if __name__ == '__main__':
    os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)
    app.run(debug=True, use_reloader=False, port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [21/Jul/2025 19:34:04] "GET / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step


INFO:werkzeug:127.0.0.1 - - [21/Jul/2025 19:34:10] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step


...

INFO:werkzeug:127.0.0.1 - - [21/Jul/2025 19:35:06] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step


INFO:werkzeug:127.0.0.1 - - [21/Jul/2025 19:35:31] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step


INFO:werkzeug:127.0.0.1 - - [21/Jul/2025 19:39:05] "POST /predict HTTP/1.1" 200 -
